<!-- nav -->
[← 4 · Names](04_Names.ipynb) · [Home](../../README.md) · [6 · Calls, members and subscripts →](06_Calls_Members_And_Subscripts.ipynb)

# 5 · Operators

`a + b * c` is not a string of seven characters to a dialect: it is a tree, an addition whose right operand is a
multiplication. This tutorial builds that tree in each language's dialect, in the builders' own calls, and checks that
each dialect's `Text.ToText` writes the language's own text back. Then it shows what the tree decides and the text
does not: where the parentheses go, which operator binds tighter in which language, how a chain of comparisons is held,
which operations a dialect defines, and what each language's arithmetic gives.

The store is the toolkit's `dialects()`, every dialect installed and defined. `S` is its holders' namespace,
`store.Expressions`, and `amd64` the System V ABI's dialect, which C's terms are read under (tutorial 2). `variable`
writes a name, as tutorial 4 showed, wherever a dialect's term is written:

In [1]:
from mbse.Expressions import Translators
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.Framework import Domains, Terms
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Python3
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.TypeScript import TypeScript5
from toolkit import dialects, show

store = dialects()
S = store.Expressions
Term = S.Framework.Term
amd64 = store.Instances.Expressions.SystemV.AMD64
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))

<a id="T05.1"></a>
## 05.1 · a + b * c is a tree

Patterns, the core vocabulary, has no operators: every operation is a `call` of an operation it names, `callable`, on
its `arguments`, each a term of the framework's union, here a Patterns term. `a + b * c` is `add(a, mul(b, c))`:

In [2]:
with store.transaction():
    patterns = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("add").arguments([
        lambda t: t.patterns(variable("a")),
        lambda t: t.patterns(lambda d: d.call(lambda m: m.callable("mul").arguments([
            lambda t: t.patterns(variable("b")), lambda t: t.patterns(variable("c"))])))]))))
for term in Terms.walk(patterns):
    form = Terms.form(term)
    print(f"{form.kind:8}", form.attributes.get("callable") or form.attributes.get("name"), len(form.arguments))

call     add 2
variable a 0
call     mul 2
variable b 0
variable c 0


`Terms.walk` visits the tree parents first, each term before its arguments, in order; `Terms.form` reads each one's
kind, its attributes and its arguments. `Terms.fold` goes the other way, children first: it calls a function once per
term with what it gave for that term's arguments, so a tree's depth and its text are each one short function:

In [3]:
def text(term, arguments):
    form = Terms.form(term)
    return form.attributes["name"] if form.kind == "variable" else f"{form.attributes['callable']}({', '.join(arguments)})"


print(Terms.fold(patterns, lambda term, depths: 1 + max(depths, default=0)))
print(Terms.fold(patterns, text))
assert Terms.fold(patterns, text) == show(patterns)

3
add(a, mul(b, c))


The toolkit's `show` is that fold, for every kind Patterns has; tutorial 13 writes analyses over constraints this way.

<a id="T05.2"></a>
## 05.2 · The same tree in each language

Each language's dialect has operators, and holds the tree as that language's own grammar does: Python3's `binop`
(Python's `ast.BinOp`, its operator `op`), TypeScript5's and C's `binaryexpression` (`left`, `operator`, `right`),
SystemVerilog's `binaryexpression` too, its names `nameexpression`s of an identifier, Excel's `infix`, MATLAB's and
LaTeX's `binary`. Each is built here in its builders' calls, and each prints the language's own text:

In [4]:
with store.transaction():
    trees = {
        "Python3": (Python3.Text, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.binop(lambda b: b
            .left(variable("a")).op("+")
            .right(lambda r: r.binop(lambda m: m.left(variable("b")).op("*").right(variable("c")))))))),
        "TypeScript5": (TypeScript5.Text, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.binaryexpression(lambda b: b
                .left(lambda l: l.identifier(lambda i: i.name("a"))).operator("+")
                .right(lambda r: r.binaryexpression(lambda m: m
                    .left(lambda l: l.identifier(lambda i: i.name("b"))).operator("*")
                    .right(lambda r: r.identifier(lambda i: i.name("c"))))))))),
        "C": (C.Text, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
            .left(variable("a")).operator("+")
            .right(lambda r: r.binaryexpression(lambda m: m.left(variable("b")).operator("*").right(variable("c"))))))
            .abi([amd64]))),
        "SystemVerilog": (SystemVerilog.Text, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.binaryexpression(lambda b: b
                .left(lambda l: l.nameexpression(lambda n: n.name(lambda i: i.identifier(lambda d: d.spelling("a")))))
                .operator("+")
                .right(lambda r: r.binaryexpression(lambda m: m
                    .left(lambda l: l.nameexpression(lambda n: n.name(lambda i: i.identifier(lambda d: d.spelling("b")))))
                    .operator("*")
                    .right(lambda r: r.nameexpression(lambda n: n.name(lambda i: i.identifier(lambda d: d.spelling("c"))))))))))),
        "Excel": (Excel.Text, S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.infix(lambda b: b
            .left(variable("a")).operator("+")
            .right(lambda r: r.infix(lambda m: m.left(variable("b")).operator("*").right(variable("c")))))))),
        "MATLAB": (Matlab.Text, S.MATLAB.Expressions.Expression(lambda e: e.root(lambda u: u.binary(lambda b: b
            .left(variable("a")).operator("+")
            .right(lambda r: r.binary(lambda m: m.left(variable("b")).operator(".*").right(variable("c")))))))),
        "LaTeX": (Latex.Text, S.LaTeX.Expressions.Expression(lambda e: e.root(lambda u: u.binary(lambda b: b
            .left(variable("a")).operator("+")
            .right(lambda r: r.binary(lambda m: m.left(variable("b")).operator("\\cdot").right(variable("c")))))))),
    }
native = {"Python3": "a + b * c", "TypeScript5": "a + b * c", "C": "a + b * c", "SystemVerilog": "a + b * c",
          "Excel": "=a + b * c", "MATLAB": "a + b .* c", "LaTeX": "a + b \\cdot c"}
for language, (text, tree) in trees.items():
    written = text.ToText(tree)
    print(f"{language:13} {written:15} {Terms.form(Terms.term(tree)).kind}")
    assert written == native[language]

Python3       a + b * c       binop
TypeScript5   a + b * c       binaryexpression
C             a + b * c       binaryexpression
SystemVerilog a + b * c       binaryexpression
Excel         =a + b * c      infix
MATLAB        a + b .* c      binary
LaTeX         a + b \cdot c   binary


Python's text is also read: `Python3.Text.FromText` parses it with Python's own `ast`, and gives the same tree as the
builders' calls (`Terms.same`). TypeScript's twin of this notebook builds it, since only Python reads Python source:

In [5]:
with store.transaction():
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("a + b * c")))
print(Python3.Text.ToText(read), Terms.same(read, trees["Python3"][1]))

a + b * c True


<a id="T05.3"></a>
## 05.3 · Parentheses are the printer's, or written

`(a + b) * c` is `mul(add(a, b), c)`: the addition is the multiplication's left operand. No term in that tree is a
parenthesis; each language's printer writes one where its precedence needs it, here in every language. C's tree, in
its builders' calls, and the others translated from Patterns' (tutorial 4), as tutorial 14 does with constraints:

In [6]:
with store.transaction():
    product = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("mul").arguments([
        lambda t: t.patterns(lambda d: d.call(lambda s: s.callable("add").arguments([
            lambda t: t.patterns(variable("a")), lambda t: t.patterns(variable("b"))]))),
        lambda t: t.patterns(variable("c"))]))))
    c = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
        .left(lambda l: l.binaryexpression(lambda s: s.left(variable("a")).operator("+").right(variable("b"))))
        .operator("*").right(variable("c")))).abi([amd64]))
assert C.Text.ToText(c) == "(a + b) * c"
for module in (Python3, C, SystemVerilog, Excel, Matlab, Latex):
    with store.transaction():
        tree = Term(Translators.between(Patterns.Expressions.DIALECT, module.Expressions.DIALECT).forward(product))
    kinds = {Terms.form(term).kind for term in Terms.walk(tree)}
    print(f"{module.Expressions.DIALECT.name():13} {module.Text.ToText(tree):17} {', '.join(sorted(kinds))}")

Python3       (a + b) * c       binop, variable
C             (a + b) * c       binaryexpression, variable
SystemVerilog (a + b) * c       binaryexpression, nameexpression
Excel         =(a + b) * c      infix, variable
MATLAB        (a + b) .* c      binary, variable
LaTeX         (a + b) \cdot c   binary, variable


Reading text, the parentheses go: Python's `ast` keeps none, so `Python3.Text.FromText` reads `a + (b * c)` as the
tree of `a + b * c`, and `(a)` as the name `a`. What the text meant is kept; how it was spelled is not:

In [7]:
with store.transaction():
    redundant = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("a + (b * c)")))
    name = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("(a)")))
print(Python3.Text.ToText(redundant), Terms.same(redundant, trees["Python3"][1]))
print(Python3.Text.ToText(name), Terms.form(Terms.term(name)).kind)

a + b * c True
a variable


Parentheses a writer wants kept are data: the dialects whose grammars hold them have a kind for them, Python3's
`parenthesized` (`value`), and TypeScript5's, C's and SystemVerilog's `parenthesizedexpression` (`expression`). Each is
printed as written, needed or not, and is not the same tree as the one without it:

In [8]:
with store.transaction():
    kept = {
        "Python3": (Python3.Text, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.binop(lambda b: b
            .left(variable("a")).op("+")
            .right(lambda r: r.parenthesized(lambda p: p.value(
                lambda v: v.binop(lambda m: m.left(variable("b")).op("*").right(variable("c")))))))))),
        "TypeScript5": (TypeScript5.Text, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.binaryexpression(lambda b: b
                .left(lambda l: l.identifier(lambda i: i.name("a"))).operator("+")
                .right(lambda r: r.parenthesizedexpression(lambda p: p.expression(
                    lambda x: x.binaryexpression(lambda m: m
                        .left(lambda l: l.identifier(lambda i: i.name("b"))).operator("*")
                        .right(lambda r: r.identifier(lambda i: i.name("c"))))))))))),
        "C": (C.Text, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
            .left(variable("a")).operator("+")
            .right(lambda r: r.parenthesizedexpression(lambda p: p.expression(
                lambda x: x.binaryexpression(lambda m: m.left(variable("b")).operator("*").right(variable("c"))))))))
            .abi([amd64]))),
    }
for language, (text, tree) in kept.items():
    print(f"{language:13} {text.ToText(tree):13} {Terms.same(tree, trees[language][1])}")
    assert text.ToText(tree) == "a + (b * c)"

Python3       a + (b * c)   False
TypeScript5   a + (b * c)   False
C             a + (b * c)   False


<a id="T05.4"></a>
## 05.4 · Precedence differs

Which operator binds tighter is each language's own, so one tree is written differently in each. A status test,
`eq(bitand(s, 4), 0)`: in Python `&` binds tighter than `==`, and no parenthesis is needed; in C and SystemVerilog
`==` binds tighter than `&`, and the printer writes one; Excel has no `&` operator for bits, but a function, `BITAND`:

In [9]:
with store.transaction():
    status = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("eq").arguments([
        lambda t: t.patterns(lambda d: d.call(lambda b: b.callable("bitand").arguments([
            lambda t: t.patterns(variable("s")),
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(4))))]))),
        lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(0))))]))))
    window = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("and").arguments([
        lambda t: t.patterns(lambda d: d.call(lambda n: n.callable("not").arguments([
            lambda t: t.patterns(lambda d: d.call(lambda g: g.callable("gt").arguments([
                lambda t: t.patterns(variable("x")),
                lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(1))))])))]))),
        lambda t: t.patterns(lambda d: d.call(lambda l: l.callable("lt").arguments([
            lambda t: t.patterns(variable("x")),
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(5))))])))]))))
print(show(status), " ", show(window))
for module in (Python3, C, SystemVerilog, Excel):
    written = []
    for tree in (status, window):
        with store.transaction():
            written.append(module.Text.ToText(Term(Translators.between(
                Patterns.Expressions.DIALECT, module.Expressions.DIALECT).forward(tree))))
    print(f"{module.Expressions.DIALECT.name():13} {written[0]:17} {written[1]}")

eq(bitand(s, 4), 0)   and(not(gt(x, 1)), lt(x, 5))
Python3       s & 4 == 0        not x > 1 and x < 5
C             (s & 4) == 0      !(x > 1) && x < 5
SystemVerilog (s & 4) == 0      !(x > 1) && x < 5
Excel         =BITAND(s, 4) = 0 =AND(NOT(x > 1), x < 5)


`not` is the other way round: Python's `not` binds looser than `>`, so `not x > 1` is `not (x > 1)`, where C's `!`
binds tighter than any binary operator, and the printer writes `!(x > 1)`. A text copied from one language to the other
would change its meaning; the tree, translated, does not.

Python's text means that tree, as `FromText` reads it and the translator takes it back to Patterns:

In [10]:
with store.transaction():
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("s & 4 == 0")))
    back = Term(Translators.between(Python3.Expressions.DIALECT, Patterns.Expressions.DIALECT).forward(read))
print(show(back), Terms.same(back, status))

eq(bitand(s, 4), 0) True


<a id="T05.5"></a>
## 05.5 · Chains and n-ary logic

Some languages' operators take more than two operands. Python's `2.0 <= x <= 8.0` is one `compare`: a `left` operand,
then its `comparisons`, each a `comparison` helper, an `op` and its `comparator`. `a and b and c` is one `boolop` of
three `values`. Each is built in its builders' calls:

In [11]:
with store.transaction():
    chain = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.compare(lambda c: c
        .left(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(2.0)).text("2.0")))
        .comparisons([
            lambda k: k.op("<=").comparator(variable("x")),
            lambda k: k.op("<=").comparator(lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(8.0)).text("8.0")))]))))
    conjunction = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.boolop(lambda b: b
        .op("and").values([variable("a"), variable("b"), variable("c")]))))
for tree in (chain, conjunction):
    form = Terms.form(Terms.term(tree))
    print(f"{Python3.Text.ToText(tree):16} {form.kind:8} {len(form.arguments)} operands")
assert (Python3.Text.ToText(chain), Python3.Text.ToText(conjunction)) == ("2.0 <= x <= 8.0", "a and b and c")

2.0 <= x <= 8.0  compare  3 operands
a and b and c    boolop   3 operands


Patterns has neither: its `le` and `and` take two arguments. Translated to Patterns, the chain is the `and` of its
links, its middle operand in both, and the `boolop` binary `and`s, the first nested, as Python evaluates them:

In [12]:
for tree in (chain, conjunction):
    with store.transaction():
        print(show(Term(Translators.between(Python3.Expressions.DIALECT, Patterns.Expressions.DIALECT).forward(tree))))

and(le(2.0, x), le(x, 8.0))
and(and(a, b), c)


Python reads both texts to the same trees:

In [13]:
with store.transaction():
    read = [S.Python.Python3.Expression(lambda e, source=source: e.root(Python3.Text.FromText(source)))
            for source in ("2.0 <= x <= 8.0", "a and b and c")]
print(Terms.same(read[0], chain), Terms.same(read[1], conjunction))

True True


<a id="T05.6"></a>
## 05.6 · The operations a dialect defines are data

What Patterns' `callable` may name is not in its code but in the store: Patterns' dialect is a singleton, found by its
global name, which names its **operations**, written by its `define`. `Domains.operations` reads them by name, in the
order the dialect defines them:

In [14]:
dialect = store.Instances.Expressions.Patterns.Expressions  # the dialect, by its global name
operations = Domains.operations(dialect)
print(", ".join(f"{name}/{Domains.arity(operation)}" for name, operation in operations.items()))
add = operations["add"]  # an operation, by its name
print(add.name, ", ".join(add.pattern.variables), add.result.name)

get/2, has/2, eq/2, ne/2, lt/2, le/2, gt/2, ge/2, and/2, or/2, not/1, implies/2, add/2, sub/2, mul/2, neg/1, bitand/2, bitor/2, bitxor/2, bitnot/1, shl/2, shr/2, count/1, item/2, in/2, sum/1, min/1, max/1, unique/1, entries/2
Expressions.Patterns.Expressions.add a, b Expressions.Patterns.Expressions.Number


Each operation is an expression schema, `Expressions.Patterns.Expressions.add`: its pattern's variables are its
arguments, in order, so `Domains.arity` is their number, and its `result` is what it returns, here a number. They are
the core vocabulary every binding evaluates. A call of any other operation is refused by validation, which reads them,
not code: here `div`, which Patterns does not define (division has no single meaning across languages: 05.7), while
05.1's tree, of `add` and `mul`, passes:

In [15]:
with store.transaction():
    quotient = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("div").arguments([
        lambda t: t.patterns(variable("a")), lambda t: t.patterns(variable("b"))]))))
print(Patterns.Expressions.DIALECT.validate(quotient, bound=["a", "b"]))
print(Patterns.Expressions.DIALECT.validate(patterns, bound=["a", "b", "c"]))

["'div' is not an operation Patterns defines"]
[]


The languages' dialects name no operations: their operators are their grammars' kinds, `binop` and the rest, each
operator a string their printers and evaluators know, as 05.2 showed:

In [16]:
print(len(Domains.operations(store.Instances.Expressions.Python.Python3)),
      len(Domains.operations(store.Instances.Expressions.C.Expressions)))

0 0


<a id="T05.7"></a>
## 05.7 · Each language's arithmetic

The same tree can give different values in different languages, because each language's `+` and `/` act on its own
domains (tutorial 2). `2147483647 + 1`, in Patterns, Python and C, each built in its builders' calls, the C one read
under the System V ABI:

In [17]:
with store.transaction():
    total = {
        "Patterns": S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("add").arguments([
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(2147483647)))),
            lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(1))))])))),
        "Python3": S.Python.Python3.Expression(lambda e: e.root(lambda u: u.binop(lambda b: b
            .left(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(2147483647)))).op("+")
            .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(1))))))),
        "C": S.C.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
            .left(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(2147483647)))).operator("+")
            .right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(1)))))).abi([amd64])),
    }
assert Python3.Text.ToText(total["Python3"]) == C.Text.ToText(total["C"]) == "2147483647 + 1"
print(Patterns.Simplifiers.OfAny(total["Patterns"]).evaluate(), Python3.Simplifiers.OfAny(total["Python3"]).evaluate())
try:
    C.Simplifiers.OfAny(total["C"]).evaluate()
except OverflowError as error:
    print(type(error).__name__, error)

2147483648 2147483648
OverflowError signed overflow in +: 2147483648 does not fit 'int'


Patterns' and Python's integers are ISO/IEC 11404's `integer`, unbounded; C's `int` is the ABI's, 32 bits, and its
evaluator raises where C's behavior is undefined, rather than wrap. The domains say so, read from the store:

In [18]:
for language, holder in (("Python3", store.Instances.Expressions.Python.Python3), ("C", amd64)):
    domain = Domains.domains(holder)["int"]
    print(f"{language:8}", Domains.kind_of(domain)[1], Domains.properties(domain))
print(C.Expressions.DIALECT.infer(total["C"]).name())

Python3  integer {'signed': True, 'overflow': 'raise', 'ordered': True}
C        integer {'width': 32, 'signed': True, 'overflow': 'raise', 'least': -2147483648, 'greatest': 2147483647, 'ordered': True}
int


Division differs too. Python's `//` rounds toward negative infinity, C's `/` on integers toward zero; each is its own
operator, in its own dialect, and Patterns defines neither (05.6):

In [19]:
with store.transaction():
    floor = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.binop(lambda b: b
        .left(lambda l: l.unaryop(lambda n: n.op("-").operand(lambda o: o.literal(lambda r: r.value(lambda v: v.as_scalar(7))))))
        .op("//").right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(2)))))))
    truncated = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.binaryexpression(lambda b: b
        .left(lambda l: l.unaryexpression(lambda n: n.operator("-").operand(
            lambda o: o.literal(lambda r: r.value(lambda v: v.as_scalar(7))))))
        .operator("/").right(lambda r: r.literal(lambda r: r.value(lambda v: v.as_scalar(2)))))).abi([amd64]))
print(f"{Python3.Text.ToText(floor):8}", Python3.Simplifiers.OfAny(floor).evaluate())
print(f"{C.Text.ToText(truncated):8}", C.Simplifiers.OfAny(truncated).evaluate())

-7 // 2  -4
-7 / 2   -3


Python reads its two texts, and they give the same values:

In [20]:
with store.transaction():
    read = [S.Python.Python3.Expression(lambda e, source=source: e.root(Python3.Text.FromText(source)))
            for source in ("2147483647 + 1", "-7 // 2")]
print([Python3.Simplifiers.OfAny(tree).evaluate() for tree in read])

[2147483648, -4]


## Why it works this way

- **The tree is the meaning, the text is one writing of it.** Precedence, parentheses and chains are each language's
  way of writing a tree; tools read and rewrite trees, and only printers and parsers deal in text
  ([PRI-006: text is one more dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-006)).
- **Each language's operators, as its grammar has them.** A dialect holds `binop` or `binaryexpression` because Python's
  `ast` and C's grammar do, so a model is the source's alternative representation, not a paraphrase
  ([CAP-226: complete expression dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-226.md)).
- **A neutral vocabulary is data.** Patterns' operations are stored, so a program learns what the core vocabulary can
  say without its code, and validation refuses what it cannot.

---

<!-- nav -->
[← 4 · Names](04_Names.ipynb) · [Home](../../README.md) · [6 · Calls, members and subscripts →](06_Calls_Members_And_Subscripts.ipynb)

<!-- nav -->
[← 4 · Names](04_Names.ipynb) · [Home](../../README.md) · [6 · Calls, members and subscripts →](06_Calls_Members_And_Subscripts.ipynb)